In [120]:
import os
import requests
import json
import textwrap
from pydantic import BaseModel, Field, TypeAdapter, ValidationError
from typing import List
import torch

In [121]:
API_URL = "http://localhost:8000/generate"


In [122]:
print(torch.cuda.is_available())

True


In [123]:
class ExtMetric(BaseModel):
    extracted_text: str = Field(description="The word or phrase from the text")
    metric_level: str = Field(description="Action, Outcome, or Unsure")
    metric_class: str = Field(description="Area, Emissions, Spending, etc.")
    confidence: float = Field(default=0.0, description="Confidence score between 0 and 1")
    #false_positive: bool = Field(default=False, description="Whether the extracted metric is a false positive")
    #false_negative: bool = Field(default=False, description="Whether the extracted metric is a false negative")
    


In [124]:
def build_prompt(text):
    return f"""
You are a strict information extraction system.

TASK:
Extract ALL quantitative or policy-relevant metrics from the text.

Only extract metrics that are:
- numeric values
- measurements
- financial values
- percentages
- emissions
- counts tied to infrastructure or policy actions

IGNORE:
- unrelated statistics
- descriptive text without measurable policy relevance
- demographic statistics unrelated to environmental policy
- student populations
- general temperature observations
- generic technical descriptions without policy importance

DEFINITIONS:

Action:
A policy intervention, implementation activity,
restoration effort, construction target,
spending allocation, or operational commitment.

Outcome:
A measurable environmental or policy result caused by actions,
such as emissions reduction, biodiversity improvement,
or water quality change.

Unsure:
Use only if classification is genuinely ambiguous.

VALID metric_class VALUES:
- Area
- Emissions
- Site Status
- Spending
- Policy Action
- Knowledge Resource
- Practical Resource
- Environment Quality
- Miscellaneous

OUTPUT RULES:
- Return ONLY valid JSON
- Return a LIST
- No explanations
- No markdown
- No extra commentary

EXAMPLES:

TEXT:
"The project will restore 12,000 hectares of peatland."

OUTPUT:
[
  {{
    "extracted_text": "12,000 hectares of peatland",
    "metric_level": "Action",
    "metric_class": "Area",
    "confidence": 0.96
  }}
]

TEXT:
"A budget of €50 million was allocated."

OUTPUT:
[
  {{
    "extracted_text": "€50 million",
    "metric_level": "Action",
    "metric_class": "Spending",
    "confidence": 0.98
  }}
]

TEXT:
"Emissions will decline by 2.4 million tCO2e."

OUTPUT:
[
  {{
    "extracted_text": "2.4 million tCO2e",
    "metric_level": "Outcome",
    "metric_class": "Emissions",
    "confidence": 0.97
  }}
]

TEXT:
"Student enrollment reached 400,000."

OUTPUT:
[]

TEXT:
"We require installation of 450 plastic piling dams."

OUTPUT:
[
  {{
    "extracted_text": "450 plastic piling dams",
    "metric_level": "Action",
    "metric_class": "Policy Action",
    "confidence": 0.88
  }}
]

TEXT:
{text}
"""

In [125]:
def call_model(prompt: str):
    payload = {
        "prompt": prompt,
        "max_new_tokens": 1024,
        "temperature": 0.1,
        "top_p": 0.9   
    }

    response = requests.post(API_URL, json=payload)
    
    print("STATUS CODE:", response.status_code)
    print("RAW SERVER RESPONSE:", response.text)

    if response.status_code != 200:
        print("SERVER ERROR:")
        print(response.text)
        return ""

    data = response.json()

    if "response" not in data:
        print("SERVER ERROR RESPONSE:", data)
        return ""

    raw_text = data["response"]

    print("OUTPUT METRICS:\n", raw_text)

    return raw_text.strip()

In [ ]:
#extract JSON block

In [38]:
"""def parse_output(raw_text: str):
    try:
        
        start = raw_text.find("[")
        end = raw_text.rfind("]") + 1
        json_str = raw_text[start:end]

        data = json.loads(json_str)

        return [ExtMetric(**item) for item in data]

    except Exception as e:
        print("Parsing failed:", e)
        print("Raw output:", raw_text)
        return []
"""

'def parse_output(raw_text: str):\n    try:\n\n        start = raw_text.find("[")\n        end = raw_text.rfind("]") + 1\n        json_str = raw_text[start:end]\n\n        data = json.loads(json_str)\n\n        return [ExtMetric(**item) for item in data]\n\n    except Exception as e:\n        print("Parsing failed:", e)\n        print("Raw output:", raw_text)\n        return []\n'

In [126]:
def parse_output(raw):
    import json
    import re
    try:
        data = json.loads(raw)

        if isinstance(data, dict):
            data = [data]

    except:
        matches = re.findall(r'\{.*?\}', raw, re.DOTALL)             # extract multiple json objects manually
        data = []

        for match in matches:
            try:
                obj = json.loads(match)
                data.append(obj)
            except:
                pass

    cleaned = []
    valid_levels = [
        "Action",
        "Outcome",
        "Unsure"
    ]

    valid_classes = [
        "Area",
        "Emissions",
        "Site Status",
        "Spending",
        "Policy Action",
        "Knowledge Resource",
        "Practical Resource",
        "Environment Quality",
        "Miscellaneous"
    ]

    for item in data:

        extracted_text = item.get(
            "extracted_text",
            ""
        ).strip()

        metric_level = item.get(
            "metric_level",
            "Unsure"
        ).strip()

        metric_class = item.get(
            "metric_class",
            "Miscellaneous"
        ).strip()
        
        confidence = float(
            item.get("confidence", 0.0)
        )
        
        #false_positive = bool(item.get(
            #"false_positive",False
        #))

        #false_negative = bool(item.get(
         #   "false_negative",False
        #))
        
        if metric_level not in valid_levels:
            metric_level = "Unsure"

        if metric_class not in valid_classes:
            metric_class = "Miscellaneous"

        
        bad_phrases = [
        "student enrollment",
        "student population",
        "temperature of the site",
        "generic infrastructure",
        "technical specifications"
        ]

        if any(
            phrase in extracted_text.lower()
            for phrase in bad_phrases
        ):
            continue

        if extracted_text:

            cleaned.append(
                ExtMetric(
                    extracted_text=extracted_text,
                    metric_level=metric_level,
                    metric_class=metric_class,
                    confidence=confidence,
                    #false_positive=false_positive,
                    #false_negative=false_negative
                )
            )

    return cleaned

In [ ]:
"""def parse_output(raw):
    try:
        # extract JSON block 
        start = raw.find("[")
        end = raw.rfind("]") + 1
        clean = raw[start:end]

        return json.loads(clean)
    except Exception as e:
        print("Parsing failed:", e)
        print("Raw output:", raw)
        return []
"""

In [ ]:
"""gen_test_text = """
Peatland Restoration and Climate Action Policy 2026
By 2030, the government commits to the rewetting of 50,000 hectares.
We anticipate a reduction of 4.2 million tCO2e by 2040.
A budget of €120 million has been allocated.
"""


In [127]:
gen_test_text = textwrap.dedent("""
Peatland Restoration and Climate Action Policy 2026
1. Land Management and Conservation Targets
The Department aims to enhance the resilience of national carbon sinks through active intervention.
By 2030, the government commits to the rewetting of 50,000 hectares of degraded raised bogs.
This target focuses on the SAC-designated North-Western Complex, which has been prioritized for immediate hydrological restoration.
Conversely, the urbanization of suburban zones has increased by 12% in the last decade, a trend that must be decoupled from environmental planning.
2. Emissions Reductions and Air Quality
A primary objective of this framework is the mitigation of greenhouse gas release.
We anticipate a total reduction of 4.2 million tCO2e by 2040 specifically from peatland re-vegetation.
This is distinct from our national goal to reduce passenger vehicle emissions by 15%, which is handled under the Transport Directive.
Furthermore, the Peatland Carbon Assessment Report will be published annually to track progress.
3. Financial Allocation and Scheme Development
To support rural communities, the Minister has authorized the Peatland Agri-Environmental Scheme (PAES) to provide direct support to landowners.
A dedicated budget of €120 million has been ring-fenced for this purpose.
In comparison, the Department of Education has noted that student enrollment has reached 600,000 pupils, requiring a separate infrastructure budget not covered under this environmental mandate.
4. Technical Specifications and Restoration Depth
Restoration success is measured by the stability of the water table.
We require the installation of 450 plastic piling dams per site to ensure water levels remain within 10cm of the surface.
During the pilot phase, the average temperature of the site was recorded at 18°C, which, while noted by researchers, is not a metric for policy delivery or outcome.
""")

In [128]:
prompt = build_prompt(gen_test_text)

raw_output = call_model(prompt)

print("\nRAW OUTPUT:\n")
print(raw_output)

metrics = parse_output(raw_output)

print("\nPARSED METRICS:\n")

if not metrics:
    print("No metrics extracted.")

else:
    for m in metrics:
        print(
            f"{m.extracted_text} | "
            f"{m.metric_level} | "
            f"{m.metric_class}"
            f"confidence={m.confidence}"
        )

STATUS CODE: 200
RAW SERVER RESPONSE: {"response":"[\n  {\n    \"extracted_text\": \"50,000 hectares of degraded raised bogs\",\n    \"metric_level\": \"Action\",\n    \"metric_class\": \"Area\",\n    \"confidence\": 0.96\n  },\n  {\n    \"extracted_text\": \"12%\",\n    \"metric_level\": \"Action\",\n    \"metric_class\": \"Percentage\",\n    \"confidence\": 0.96\n  },\n  {\n    \"extracted_text\": \"4.2 million tCO2e\",\n    \"metric_level\": \"Outcome\",\n    \"metric_class\": \"Emissions\",\n    \"confidence\": 0.96\n  },\n  {\n    \"extracted_text\": \"€120 million\",\n    \"metric_level\": \"Action\",\n    \"metric_class\": \"Spending\",\n    \"confidence\": 0.96\n  },\n  {\n    \"extracted_text\": \"600,000 pupils\",\n    \"metric_level\": \"Action\",\n    \"metric_class\": \"Count tied to Infrastructure or Policy Actions\",\n    \"confidence\": 0.96\n  },\n  {\n    \"extracted_text\": \"450 plastic piling dams per site\",\n    \"metric_level\": \"Action\",\n    \"metric_class\"